In [ ]:
'''Develop a stacking ensemble method combining multiple diverse classifiers to 
boost predictive accuracy and robustness in university admission predictions.
'''

In [1]:

# import libraries required for data processing and model building

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)

print("all libraries imported successfully")


all libraries imported successfully


In [2]:

# load the university admission dataset

df = pd.read_csv("Adm.csv")

print("dataset loaded successfully")
print("dataset shape:", df.shape)

df.head()


dataset loaded successfully
dataset shape: (1000, 12)


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit,Application_DateTime,University_Name,region
0,1,337,118,4,4.5,4.5,9.65,1,0.92,2022-01-01 09:00:00,Indian Institute of Technology Kharagpur,East
1,2,324,107,4,4.0,4.5,8.87,1,0.76,2022-01-02 09:00:00,National Institute of Technology Karnataka,South
2,3,316,104,3,3.0,3.5,8.00,1,0.72,2022-01-04 09:00:00,Gujarat University,West
3,4,322,110,3,3.5,2.5,8.67,1,0.80,2022-01-06 09:00:00,Jamia Millia Islamia,North
4,5,314,103,2,2.0,3.0,8.21,0,0.65,2022-01-08 09:00:00,Jamia Millia Islamia,North


In [3]:

# check dataset information and understand the available columns

print("column names:")
print(df.columns.tolist())

print("\ndataset information:")
df.info()

print("\nmissing values:")
print(df.isnull().sum())

print("\nstatistical summary:")
df.describe(include="all")


column names:
['Serial No.', 'GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit', 'Application_DateTime', 'University_Name', 'region']

dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Serial No.            1000 non-null   int64  
 1   GRE Score             1000 non-null   int64  
 2   TOEFL Score           1000 non-null   int64  
 3   University Rating     1000 non-null   int64  
 4   SOP                   1000 non-null   float64
 5   LOR                   1000 non-null   float64
 6   CGPA                  1000 non-null   float64
 7   Research              1000 non-null   int64  
 8   Chance of Admit       1000 non-null   float64
 9   Application_DateTime  1000 non-null   object 
 10  University_Name       1000 non-null   object 
 11  region              

,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit,Application_DateTime,University_Name,region
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.00000,1000,1000,1000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1000,49,6
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-01-01 09:00:00,National Institute of Technology Warangal,South
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,32,379
mean,500.500000,315.965000,106.564000,3.015000,3.393500,3.406700,8.58979,0.503000,0.71777,NaN,NaN,NaN
std,288.819436,13.415653,7.634277,1.317768,1.027279,0.891864,0.61217,0.500241,0.14591,NaN,NaN,NaN
min,1.000000,290.000000,92.000000,1.000000,0.900000,0.900000,6.77000,0.000000,0.32000,NaN,NaN,NaN
25%,250.750000,305.000000,101.000000,2.000000,2.500000,3.000000,8.18000,0.000000,0.63750,NaN,NaN,NaN
50%,500.500000,316.000000,107.000000,3.000000,3.500000,3.500000,8.58000,1.000000,0.72000,NaN,NaN,NaN
75%,750.250000,326.250000,113.000000,4.000000,4.000000,4.000000,9.07000,1.000000,0.82000,NaN,NaN,NaN


In [4]:

# clean unnecessary columns and prepare useful application date features

df["Application_DateTime"] = pd.to_datetime(
    df["Application_DateTime"], errors="coerce"
)

df["Application_Year"] = df["Application_DateTime"].dt.year
df["Application_Month"] = df["Application_DateTime"].dt.month

df.drop(
    columns=["Serial No.", "Application_DateTime"],
    inplace=True
)

print("data cleaning completed")
df.head()


data cleaning completed


,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit,University_Name,region,Application_Year,Application_Month
0,337,118,4,4.5,4.5,9.65,1,0.92,Indian Institute of Technology Kharagpur,East,2022,1
1,324,107,4,4.0,4.5,8.87,1,0.76,National Institute of Technology Karnataka,South,2022,1
2,316,104,3,3.0,3.5,8.00,1,0.72,Gujarat University,West,2022,1
3,322,110,3,3.5,2.5,8.67,1,0.80,Jamia Millia Islamia,North,2022,1
4,314,103,2,2.0,3.0,8.21,0,0.65,Jamia Millia Islamia,North,2022,1


In [5]:

# convert admission chance into high and low admission categories

df["Admission_Result"] = (
    df["Chance of Admit"] >= 0.50
).astype(int)

print("0 = low admission chance")
print("1 = high admission chance")

print("\nadmission category distribution:")
print(df["Admission_Result"].value_counts())


0 = low admission chance
1 = high admission chance

admission category distribution:
Admission_Result
1    906
0     94
Name: count, dtype: int64


In [6]:

# separate student details as input and admission result as target

X = df.drop(columns=["Chance of Admit", "Admission_Result"])
y = df["Admission_Result"]

print("input features:", X.shape)
print("target variable:", y.shape)

print("\ninput columns:")
print(X.columns.tolist())


input features: (1000, 11)
target variable: (1000,)

input columns:
['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'University_Name', 'region', 'Application_Year', 'Application_Month']


In [7]:

# identify numerical and categorical columns for preprocessing

numeric_cols = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("numerical columns:")
print(numeric_cols)

print("\ncategorical columns:")
print(categorical_cols)


numerical columns:
['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Application_Year', 'Application_Month']

categorical columns:
['University_Name', 'region']


In [8]:

# fill missing values, scale numerical features and encode categorical features

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

print("data preprocessing pipeline created successfully")


data preprocessing pipeline created successfully


In [9]:

# divide data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("training records:", X_train.shape[0])
print("testing records:", X_test.shape[0])

print("\ntraining class distribution:")
print(y_train.value_counts())


training records: 800
testing records: 200

training class distribution:
Admission_Result
1    725
0     75
Name: count, dtype: int64


In [10]:

# create different classifiers to learn admission patterns

base_models = [

    ("logistic", Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])),

    ("decision_tree", Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=42
        ))
    ])),

    ("random_forest", Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=8,
            class_weight="balanced",
            random_state=42
        ))
    ])),

    ("knn", Pipeline([
        ("prep", preprocessor),
        ("model", KNeighborsClassifier(
            n_neighbors=7,
            weights="distance"
        ))
    ])),

    ("svm", Pipeline([
        ("prep", preprocessor),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ]))
]

print("five different base classifiers created")

for name, model in base_models:
    print(name)


five different base classifiers created
logistic
decision_tree
random_forest
knn
svm


In [11]:

# combine base classifiers using logistic regression as the meta learner

stack_model = StackingClassifier(
    estimators=base_models,

    final_estimator=LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),

    cv=5,
    stack_method="predict_proba",
    n_jobs=-1
)

print("stacking ensemble architecture created")
print("base models: 5")
print("meta learner: logistic regression")
print("cross validation folds: 5")


stacking ensemble architecture created
base models: 5
meta learner: logistic regression
cross validation folds: 5


In [12]:

# train the stacking ensemble using student admission records

stack_model.fit(X_train, y_train)

print("stacking ensemble training completed successfully")


stacking ensemble training completed successfully


In [13]:

# predict admission categories and calculate admission probabilities

y_pred = stack_model.predict(X_test)

y_prob = stack_model.predict_proba(X_test)[:, 1]

print("predictions completed")

print("\nfirst ten predictions:")
print(y_pred[:10])

print("\nadmission probabilities:")
print(np.round(y_prob[:10], 3))


predictions completed

first ten predictions:
[0 0 1 1 0 1 0 1 1 1]

admission probabilities:
[0.456 0.373 0.557 0.705 0.117 0.567 0.249 0.669 0.578 0.678]


In [14]:

# calculate important performance metrics for the stacking model

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, zero_division=0
)

recall = recall_score(
    y_test, y_pred, zero_division=0
)

f1 = f1_score(
    y_test, y_pred, zero_division=0
)

balanced_acc = balanced_accuracy_score(y_test, y_pred)

auc = roc_auc_score(y_test, y_prob)

performance = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision",
        "recall",
        "f1 score",
        "balanced accuracy",
        "roc auc"
    ],

    "score": [
        accuracy,
        precision,
        recall,
        f1,
        balanced_acc,
        auc
    ]
})

print("stacking model performance:")
print(performance.round(4).to_string(index=False))


stacking model performance:
           metric  score
         accuracy 0.6400
        precision 0.9580
           recall 0.6298
         f1 score 0.7600
balanced accuracy 0.6833
          roc auc 0.7813


In [15]:

# compare individual classifiers with the stacking ensemble

comparison = []

for name, model in base_models:

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)

    comparison.append({
        "model": name,
        "accuracy": accuracy_score(y_test, prediction),
        "precision": precision_score(
            y_test, prediction, zero_division=0
        ),
        "recall": recall_score(
            y_test, prediction, zero_division=0
        ),
        "f1_score": f1_score(
            y_test, prediction, zero_division=0
        )
    })

comparison.append({
    "model": "stacking ensemble",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1
})

comparison_df = pd.DataFrame(comparison)

print("individual model vs stacking model:")
comparison_df.round(4)


individual model vs stacking model:


,model,accuracy,precision,recall,f1_score
0,logistic,0.635,0.9355,0.6409,0.7607
1,decision_tree,0.530,0.9677,0.4972,0.6569
2,random_forest,0.915,0.9362,0.9724,0.9539
3,knn,0.910,0.9095,1.0000,0.9526
4,svm,0.810,0.9387,0.8453,0.8895
5,stacking ensemble,0.640,0.9580,0.6298,0.7600


In [16]:

# evaluate the consistency of the stacking model using five fold validation

cv_scores = cross_val_score(
    stack_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

print("cross validation scores:")
print(np.round(cv_scores, 4))

print("\naverage macro f1 score:")
print(round(cv_scores.mean(), 4))

print("\nstandard deviation:")
print(round(cv_scores.std(), 4))


cross validation scores:
[0.5022 0.4297 0.4997 0.5042 0.4589]

average macro f1 score:
0.479

standard deviation:
0.0298


In [17]:

# display correct and incorrect admission predictions

cm = confusion_matrix(y_test, y_pred)

print("confusion matrix:")
print(cm)

print("\ntrue negative:", cm[0, 0])
print("false positive:", cm[0, 1])
print("false negative:", cm[1, 0])
print("true positive:", cm[1, 1])


confusion matrix:
[[ 14   5]
 [ 67 114]]

true negative: 14
false positive: 5
false negative: 67
true positive: 114


In [18]:

# display precision recall and f1 score for both admission categories

print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "low admission chance",
        "high admission chance"
    ],
    zero_division=0
))


                       precision    recall  f1-score   support

 low admission chance       0.17      0.74      0.28        19
high admission chance       0.96      0.63      0.76       181

             accuracy                           0.64       200
            macro avg       0.57      0.68      0.52       200
         weighted avg       0.88      0.64      0.71       200



In [19]:

# compare actual admission categories with predicted results

prediction_result = pd.DataFrame({
    "actual_result": y_test.iloc[:10].values,
    "predicted_result": y_pred[:10],
    "admission_probability": y_prob[:10].round(3)
})

prediction_result["predicted_category"] = np.where(
    prediction_result["predicted_result"] == 1,
    "high chance",
    "low chance"
)

prediction_result["prediction_status"] = np.where(
    prediction_result["actual_result"] ==
    prediction_result["predicted_result"],
    "correct",
    "incorrect"
)

prediction_result


,actual_result,predicted_result,admission_probability,predicted_category,prediction_status
0,1,0,0.456,low chance,incorrect
1,0,0,0.373,low chance,correct
2,1,1,0.557,high chance,correct
3,1,1,0.705,high chance,correct
4,0,0,0.117,low chance,correct
5,1,1,0.567,high chance,correct
6,1,0,0.249,low chance,incorrect
7,1,1,0.669,high chance,correct
8,1,1,0.578,high chance,correct
9,1,1,0.678,high chance,correct


In [20]:

# summarize the final stacking ensemble performance

print("model: stacking ensemble")
print("base classifiers: 5")
print("meta learner: logistic regression")
print("training records:", len(X_train))
print("testing records:", len(X_test))

print("\naccuracy:", round(accuracy, 4))
print("precision:", round(precision, 4))
print("recall:", round(recall, 4))
print("f1 score:", round(f1, 4))
print("balanced accuracy:", round(balanced_acc, 4))
print("roc auc:", round(auc, 4))

print("average cross validation macro f1:",
      round(cv_scores.mean(), 4))

print("\nstacking ensemble evaluation completed")


model: stacking ensemble
base classifiers: 5
meta learner: logistic regression
training records: 800
testing records: 200

accuracy: 0.64
precision: 0.958
recall: 0.6298
f1 score: 0.76
balanced accuracy: 0.6833
roc auc: 0.7813
average cross validation macro f1: 0.479

stacking ensemble evaluation completed
